# 01 — Exploration du jeu de données

**Projet Air Paradis — prédiction du sentiment d'un tweet**

---

## Objectif de ce notebook

Ce notebook est le **point de départ obligatoire** du projet. Il remplit deux rôles :

1. **Comprendre les données** avant de modéliser quoi que ce soit. On ne choisit pas une
   architecture de modèle au hasard : la longueur des tweets, l'équilibre des classes et le
   vocabulaire employé déterminent directement les paramètres des modèles suivants.

2. **Produire le fichier de travail partagé.** Le nettoyage des 1,6 million de tweets est fait
   ici, une seule fois, et sauvegardé sur le disque. Les trois notebooks de modélisation
   repartent tous de ce même fichier — c'est ce qui rend la comparaison des trois approches
   honnête.

> **À exécuter avant les notebooks 02, 03 et 04.**

## 1. Préparation de l'environnement

In [ ]:
# ---------------------------------------------------------------------------
# Rendre le package src/ importable depuis le notebook.
#
# Un notebook s'exécute depuis le dossier notebooks/, alors que le code partagé
# se trouve dans src/, un dossier au-dessus. On remonte donc l'arborescence
# jusqu'à trouver la racine du projet (repérée par la présence de pytest.ini),
# puis on l'ajoute au chemin de recherche de Python.
#
# On procède ainsi plutôt qu'avec un ".." écrit en dur : le notebook fonctionne
# alors quel que soit l'endroit d'où Jupyter a été lancé.
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

RACINE_PROJET = next(
    dossier
    for dossier in [Path.cwd(), *Path.cwd().parents]
    if (dossier / "pytest.ini").exists()
)

if str(RACINE_PROJET) not in sys.path:
    sys.path.insert(0, str(RACINE_PROJET))

print(f"Racine du projet : {RACINE_PROJET}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import config
from src.data_loader import charger_donnees_brutes, preparer_donnees
from src.preprocessing import nettoyer_tweet, retirer_mots_vides

# Réglages d'affichage : graphiques lisibles et colonnes de texte non tronquées.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_colwidth", 120)

print("Bibliothèques chargées.")

## 2. Chargement des données brutes

Le jeu de données **Sentiment140** rassemble 1 600 000 tweets en anglais collectés en 2009.

Son étiquetage est *automatique* : un tweet contenant une émoticône souriante a été classé
positif, un tweet contenant une émoticône triste a été classé négatif. Les émoticônes ont
ensuite été retirées du texte, pour que le modèle ne puisse pas « tricher » en les repérant.

Cette méthode d'étiquetage a une conséquence importante à garder en tête pour toute la suite :
**les étiquettes sont bruitées**. Un tweet ironique accompagné d'un smiley sera étiqueté
positif à tort. C'est une limite structurelle du jeu de données, et elle impose un plafond de
performance qu'aucun modèle ne pourra franchir.

In [ ]:
donnees_brutes = charger_donnees_brutes()

In [ ]:
donnees_brutes.head(10)

## 3. Qualité et équilibre du jeu de données

Trois vérifications de routine, à faire systématiquement avant de modéliser :

- **Valeurs manquantes** — un texte vide fausserait l'entraînement.
- **Doublons** — un tweet présent deux fois pourrait se retrouver à la fois dans le jeu
  d'entraînement et dans le jeu de test, ce qui gonflerait artificiellement le score final.
- **Équilibre des classes** — si une classe dominait, l'exactitude (*accuracy*) deviendrait
  une métrique trompeuse et il faudrait rééquilibrer.

In [ ]:
print("Valeurs manquantes par colonne :")
print(donnees_brutes.isna().sum())

nombre_doublons = donnees_brutes["texte"].duplicated().sum()
print(f"\nTweets en double : {nombre_doublons:,}".replace(",", " "))
print(f"Soit {100 * nombre_doublons / len(donnees_brutes):.2f} % du jeu de données")

In [ ]:
repartition = donnees_brutes["label"].value_counts().sort_index()
repartition.index = ["Négatif (0)", "Positif (1)"]

axe = repartition.plot(kind="bar", color=["#d62728", "#2ca02c"], rot=0)
axe.set_title("Répartition des sentiments dans le jeu de données")
axe.set_ylabel("Nombre de tweets")

# Afficher la valeur exacte au-dessus de chaque barre.
for index, valeur in enumerate(repartition):
    axe.text(index, valeur, f"{valeur:,}".replace(",", " "), ha="center", va="bottom")

plt.tight_layout()
plt.show()

print(repartition)

### Ce qu'on retient

Le jeu de données est **parfaitement équilibré** : 800 000 tweets de chaque classe.

Deux conséquences pratiques :

- L'**exactitude** (*accuracy*) est une métrique fiable ici. Sur un jeu déséquilibré, un
  modèle qui répondrait toujours « positif » obtiendrait un bon score sans rien apprendre ;
  ce piège n'existe pas ici.
- Le **hasard** correspond à 50 %. C'est le point de comparaison minimal : tout modèle doit
  faire nettement mieux pour avoir un intérêt.

On suivra malgré tout plusieurs métriques (précision, rappel, F1, AUC), car le besoin
métier n'est pas symétrique : pour Air Paradis, **rater un tweet négatif** (un bad buzz qui
démarre) coûte bien plus cher que signaler à tort un tweet neutre.

## 4. Longueur des tweets

Cette mesure n'est pas décorative : elle détermine directement un paramètre du modèle avancé,
`LONGUEUR_MAX_SEQUENCE`.

Un réseau de neurones récurrent traite des séquences de longueur fixe. Il faut donc choisir un
nombre de mots :

- **trop court** → les tweets longs sont tronqués, on perd de l'information ;
- **trop long** → la majorité des séquences est remplie de zéros inutiles, et le temps de
  calcul augmente pour rien.

On cherche donc la longueur qui couvre la grande majorité des tweets sans excès.

In [ ]:
# Longueur en nombre de mots, sur le texte brut.
longueurs = donnees_brutes["texte"].str.split().str.len()

print("Distribution de la longueur des tweets (en mots) :")
print(longueurs.describe().round(1))

print("\nQuantiles utiles :")
for quantile in [0.50, 0.75, 0.90, 0.95, 0.99]:
    print(f"  {quantile:.0%} des tweets font au plus {longueurs.quantile(quantile):.0f} mots")

In [ ]:
axe = sns.histplot(longueurs, bins=60, color="#1f77b4")
axe.set_title("Distribution de la longueur des tweets")
axe.set_xlabel("Nombre de mots")
axe.set_ylabel("Nombre de tweets")

# Repère visuel : la valeur retenue dans la configuration du projet.
axe.axvline(
    config.LONGUEUR_MAX_SEQUENCE,
    color="red",
    linestyle="--",
    label=f"LONGUEUR_MAX_SEQUENCE = {config.LONGUEUR_MAX_SEQUENCE}",
)
axe.legend()
plt.tight_layout()
plt.show()

couverture = (longueurs <= config.LONGUEUR_MAX_SEQUENCE).mean()
print(f"La valeur retenue couvre {couverture:.1%} des tweets sans troncature.")

## 5. Effet du nettoyage

On visualise ici, sur quelques exemples réels, ce que fait concrètement
`src/preprocessing.py` : suppression des liens et des mentions, décodage des entités HTML,
réduction des lettres répétées, retrait de la ponctuation.

La colonne « sans mots vides » montre l'étape supplémentaire réservée au modèle classique.
On vérifie au passage le point le plus important du prétraitement : **les négations
survivent**.

In [ ]:
# Un échantillon reproductible de tweets, pour illustrer le nettoyage.
exemples = donnees_brutes.sample(8, random_state=config.GRAINE_ALEATOIRE)

comparaison = pd.DataFrame({
    "Brut": exemples["texte"],
    "Nettoyé": exemples["texte"].apply(nettoyer_tweet),
    "Sans mots vides": exemples["texte"].apply(lambda t: retirer_mots_vides(nettoyer_tweet(t))),
})

comparaison.reset_index(drop=True)

In [ ]:
# Vérification ciblée du traitement de la négation.
cas_de_test = [
    "@AirParadis the flight was NOT good at all!!! #disappointed http://t.co/xyz",
    "Best crew ever, sooooo friendly &amp; helpful :)",
]

for texte in cas_de_test:
    print(f"Brut            : {texte}")
    print(f"Nettoyé         : {nettoyer_tweet(texte)}")
    print(f"Sans mots vides : {retirer_mots_vides(nettoyer_tweet(texte))}")
    print("-" * 100)

## 6. Vocabulaire caractéristique de chaque classe

Quels mots reviennent le plus souvent dans les tweets négatifs, et lesquels dans les
positifs ?

Cette analyse répond à une question préalable indispensable : **le signal existe-t-il ?**
Si les deux classes employaient exactement le même vocabulaire, un modèle de type sac de mots
serait condamné d'avance et il faudrait passer directement aux approches contextuelles.

In [ ]:
from collections import Counter

# On travaille sur un échantillon de 200 000 tweets : suffisant pour une
# tendance fiable, et bien plus rapide que sur les 1,6 million.
echantillon = donnees_brutes.sample(200_000, random_state=config.GRAINE_ALEATOIRE)

compteurs = {}
for valeur_label, nom_classe in [(0, "Négatif"), (1, "Positif")]:
    textes = echantillon.loc[echantillon["label"] == valeur_label, "texte"]
    mots = " ".join(textes.apply(lambda t: retirer_mots_vides(nettoyer_tweet(t)))).split()
    compteurs[nom_classe] = Counter(mots)

figure, axes = plt.subplots(1, 2, figsize=(14, 6))

for axe, (nom_classe, couleur) in zip(axes, [("Négatif", "#d62728"), ("Positif", "#2ca02c")]):
    plus_frequents = compteurs[nom_classe].most_common(15)
    mots = [mot for mot, _ in plus_frequents][::-1]
    effectifs = [effectif for _, effectif in plus_frequents][::-1]

    axe.barh(mots, effectifs, color=couleur)
    axe.set_title(f"15 mots les plus fréquents — tweets {nom_classe.lower()}s")
    axe.set_xlabel("Occurrences")

plt.tight_layout()
plt.show()

In [ ]:
# Mots les plus DISCRIMINANTS : présents dans une classe et rares dans l'autre.
# Le rapport des fréquences est plus parlant que la fréquence brute, car les
# mots très courants ("flight", "day") apparaissent en tête des deux classes.

LISSAGE = 50  # évite de diviser par zéro et écarte les mots trop rares

frequences_negatives = compteurs["Négatif"]
frequences_positives = compteurs["Positif"]

vocabulaire_commun = set(frequences_negatives) | set(frequences_positives)

scores = {
    mot: (frequences_positives[mot] + LISSAGE) / (frequences_negatives[mot] + LISSAGE)
    for mot in vocabulaire_commun
    if frequences_positives[mot] + frequences_negatives[mot] >= 500
}

classement = sorted(scores.items(), key=lambda couple: couple[1])

print("Mots les plus caractéristiques des tweets NÉGATIFS :")
for mot, score in classement[:15]:
    print(f"  {mot:15s} {1 / score:.1f}x plus fréquent chez les négatifs")

print("\nMots les plus caractéristiques des tweets POSITIFS :")
for mot, score in classement[-15:][::-1]:
    print(f"  {mot:15s} {score:.1f}x plus fréquent chez les positifs")

### Ce qu'on retient

Le vocabulaire des deux classes est **nettement différencié**. Des mots comme *sad*, *sorry*,
*miss*, *hate* d'un côté, *thanks*, *love*, *good*, *great* de l'autre, apparaissent plusieurs
fois plus souvent dans une classe que dans l'autre.

Le signal existe donc bel et bien dans les mots isolés : une approche par sac de mots
(TF-IDF) a toutes les chances de donner un résultat correct. Elle constituera notre point de
comparaison de référence.

Sa limite est en revanche déjà visible : cette approche ignore l'ordre des mots, donc elle ne
peut pas distinguer *« not good »* de *« good »*. C'est précisément ce que les approches 2 et
3 viendront corriger.

## 7. Préparation du fichier de travail

Dernière étape : appliquer le nettoyage aux 1,6 million de tweets et sauvegarder le résultat.

L'opération prend environ une minute. Elle n'est faite **qu'une seule fois** — les notebooks
suivants rechargent le fichier produit en moins d'une seconde.

Le format retenu est **Parquet** plutôt que CSV : environ cinq fois plus compact, beaucoup
plus rapide à relire, et il conserve les types des colonnes sans avoir à les redéclarer.

In [ ]:
donnees_preparees = preparer_donnees()
donnees_preparees.head()

In [ ]:
# Vérification du fichier produit : c'est la sortie de ce notebook, celle dont
# dépendent les trois notebooks de modélisation.
fichier = config.FICHIER_DONNEES_PREPAREES
taille_mo = fichier.stat().st_size / (1024 * 1024)

print(f"Fichier   : {fichier}")
print(f"Taille    : {taille_mo:.1f} Mo")
print(f"Tweets    : {len(donnees_preparees):,}".replace(",", " "))
print(f"Colonnes  : {list(donnees_preparees.columns)}")
print(f"\nÉquilibre des classes conservé :")
print(donnees_preparees["label"].value_counts(normalize=True).round(4))

---

## Conclusion

| Constat | Conséquence pour la suite |
|---|---|
| 1,6 million de tweets, classes parfaitement équilibrées | L'exactitude est une métrique fiable ; le hasard est à 50 % |
| Étiquetage automatique par émoticônes, donc bruité | Un plafond de performance existe, autour de 85 % selon la littérature |
| 95 % des tweets font moins de 30 mots | `LONGUEUR_MAX_SEQUENCE = 40` couvre la quasi-totalité sans gaspillage |
| Vocabulaire nettement différencié entre les classes | Une approche par sac de mots donnera déjà un résultat correct |
| L'ordre des mots n'est pas exploité par le sac de mots | Justifie les approches 2 (embeddings + réseau récurrent) et 3 (BERT) |

**Fichier produit :** `data/processed/tweets_prepares.parquet`

**Étape suivante :** `02_modele_classique.ipynb`